Advanced Features Extractor

In [ ]:
import os
import re
import ast
import time
import warnings
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
from pathlib import Path
from tqdm.auto import tqdm
import joblib

# CV Libraries
from skimage.feature import hog, local_binary_pattern, graycomatrix, graycoprops
from skimage import img_as_ubyte

warnings.filterwarnings('ignore')

import kagglehub
DATA_ROOT = '/kaggle/input/competitions/birdclef-2026'
PERCH_MODEL_PATH = kagglehub.model_download("google/bird-vocalization-classifier/tensorFlow2/bird-vocalization-classifier")

OUTPUT_DIR = '/kaggle/working'
# Settings
SAMPLE_RATE = 32_000
WINDOW_SEC = 5
WINDOW_SAMPLES = SAMPLE_RATE * WINDOW_SEC
N_WINDOWS = 3 # We extract a continuous 15s block = 3 x 5s windows
BLOCK_SAMPLES = WINDOW_SAMPLES * N_WINDOWS
MIN_RATING = 2
DEBUG_MODE = False # Set to False for full run
N_DEBUG_ROWS = 10

OUT_PARQUET_PERCH = os.path.join(OUTPUT_DIR, "features_v3_perch_discrete.parquet")
OUT_PARQUET_CV = os.path.join(OUTPUT_DIR, "features_v3_cv_discrete.parquet")

import tensorflow as tf
print(f"Loading Perch model from: {PERCH_MODEL_PATH}")
# tf.config.set_visible_devices([], 'GPU') 
birdclassifier = tf.saved_model.load(PERCH_MODEL_PATH)
perch_infer = birdclassifier.signatures["serving_default"]


def parse_filename(filename):
    """extract site/hour from filename"""
    # Fallback default values
    site = "unknown"
    hour = -1
    
    # regex to match standard birdclef 2026 format
    m = re.match(r"BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg", filename)
    if m:
        _, site_str, _, hms = m.groups()
        site = site_str
        hour = int(hms[:2])
    return site, hour

def top_k_energy_windows(y, sr, segment_sec, k=3, hop_sec=2.5):
    target_len = int(sr * segment_sec)
    if len(y) <= target_len:
        padded = np.pad(y, (0, max(0, target_len - len(y))), "constant")
        return [padded], [0.0]
    hop = max(1, int(sr * hop_sec))
    starts = list(range(0, len(y) - target_len + 1, hop))
    windows = [y[s:s + target_len] for s in starts]
    energies = np.array([np.sqrt(np.mean(w.astype(np.float64) ** 2)) for w in windows])
    order = np.argsort(-energies)[:k]
    return [windows[i] for i in order], [starts[i] / sr for i in order]
def compute_cv_features(audio_chunk):
    """convert to melspec and get cv features"""
    # melspec
    S = librosa.feature.melspectrogram(
        y=audio_chunk, sr=SAMPLE_RATE, n_mels=128, fmax=16000, hop_length=512
    )
    S_dB = librosa.power_to_db(S, ref=np.max)
    
    # Normalize to 0-255 grayscale image
    img = S_dB - S_dB.min()
    if img.max() > 0:
        img = img / img.max()
    image = img_as_ubyte(img)
    
    # hog features
    # Resize or use parameters suitable for 128x313
    hog_feats = hog(
        image, orientations=8, pixels_per_cell=(16, 16),
        cells_per_block=(1, 1), feature_vector=True
    )
    # pool HOG to basic stats
    hog_stats = [np.mean(hog_feats), np.max(hog_feats), np.std(hog_feats)]
    
    # lbp texture
    radius = 2
    n_points = 8 * radius
    lbp = local_binary_pattern(image, n_points, radius, method='uniform')
    # Histogram of LBP
    n_bins = int(lbp.max() + 1)
    lbp_hist, _ = np.histogram(lbp.ravel(), bins=n_bins, range=(0, n_bins))
    lbp_hist = lbp_hist.astype(float) / (lbp_hist.sum() + 1e-6)
    
    # glcm
    # Subsample image to save compute (quantize to 16 levels)
    image_q = (image // 16).astype(np.uint8)
    glcm = graycomatrix(image_q, distances=[1, 3], angles=[0, np.pi/4], levels=16, symmetric=True, normed=True)
    
    contrast = graycoprops(glcm, 'contrast').mean()
    dissimilarity = graycoprops(glcm, 'dissimilarity').mean()
    homogeneity = graycoprops(glcm, 'homogeneity').mean()
    energy = graycoprops(glcm, 'energy').mean()
    correlation = graycoprops(glcm, 'correlation').mean()
    asm = graycoprops(glcm, 'ASM').mean()
    
    glcm_stats = [contrast, dissimilarity, homogeneity, energy, correlation, asm]
    
    # concat all cv feats
    cv_vector = np.concatenate([hog_stats, lbp_hist, glcm_stats])
    # Replace NaNs (e.g. from GLCM correlation on silent images) with 0
    cv_vector = np.nan_to_num(cv_vector)
    return cv_vector.astype(np.float32)


def process_file(file_path):
    """process audio file"""
    filename = os.path.basename(file_path)
    site, hour = parse_filename(filename)
    
    try:
        y, sr = sf.read(file_path)
        if len(y.shape) > 1:
            y = np.mean(y, axis=1)
        if sr != SAMPLE_RATE:
            y = librosa.resample(y, orig_sr=sr, target_sr=SAMPLE_RATE)
    except Exception as e:
        print(f"Error loading {filename}: {e}")
        return []

    # Determine how many 5s windows we can actually extract (max N_WINDOWS)
    # If audio is very short, we only extract 1 window.
    # Get top-K windows k is 3
    windows, start_secs = top_k_energy_windows(y, SAMPLE_RATE, WINDOW_SEC, k=N_WINDOWS)
    actual_n_windows = len(windows)

    # Shape: [actual_n_windows, 160000]
    batch_frames = np.array(windows, dtype=np.float32)
    perch_out = perch_infer(inputs=tf.convert_to_tensor(batch_frames))
    embeddings = perch_out["embedding"].numpy()

    rows_perch = []
    rows_cv = []
    for i in range(actual_n_windows):
        curr_start_sec = start_secs[i]
        
        cv_feats = compute_cv_features(windows[i])
        
        base_meta = {
            "filename": filename,
            "start_sec": curr_start_sec,
            "site": site,
            "hour": hour
        }
        
        row_perch = base_meta.copy()
        for j in range(embeddings.shape[1]):
            row_perch[f"perch_{j}"] = embeddings[i, j]
            
        row_cv = base_meta.copy()
        for j in range(len(cv_feats)):
            row_cv[f"cv_{j}"] = cv_feats[j]
            
        rows_perch.append(row_perch)
        rows_cv.append(row_cv)
        
    return rows_perch, rows_cv
def main():
    print("Loading metadata...")
    train_df = pd.read_csv(os.path.join(DATA_ROOT, 'train.csv'))
    
    # Quality filter
    train_df = train_df[train_df['rating'] >= MIN_RATING].reset_index(drop=True)
    print(f"After quality filter: {len(train_df)} rows.")
    
    if DEBUG_MODE:
        print(f"DEBUG MODE ON: Sampling {N_DEBUG_ROWS} rows.")
        train_df = train_df.sample(N_DEBUG_ROWS, random_state=42).reset_index(drop=True)
        
    all_rows_perch = []
    all_rows_cv = []
    
    print("Extracting features (Check logs every 100 rows)...")
    for idx, row in tqdm(train_df.iterrows(), total=len(train_df), miniters=100):
        file_path = os.path.join(DATA_ROOT, 'train_audio', row['primary_label'], row['filename'])
        # Depending on Kaggle structure, it might be in train_audio/
        if not os.path.exists(file_path):
            # Try flat structure
            file_path = os.path.join(DATA_ROOT, 'train_audio', row['filename'])
            
        if os.path.exists(file_path):
            perch_extracted, cv_extracted = process_file(file_path)
            all_rows_perch.extend(perch_extracted)
            all_rows_cv.extend(cv_extracted)
            
    # Save to Parquet
    df_perch = pd.DataFrame(all_rows_perch)
    df_cv = pd.DataFrame(all_rows_cv)
    print("Extraction complete.")
    print(f"Perch shape: {df_perch.shape}, CV shape: {df_cv.shape}")
    
    df_perch.to_parquet(OUT_PARQUET_PERCH)
    df_cv.to_parquet(OUT_PARQUET_CV)
    print(f"Saved Perch features to {OUT_PARQUET_PERCH}")
    print(f"Saved CV features to {OUT_PARQUET_CV}")

if __name__ == '__main__':
    main()


2026-05-14 12:17:32.207085: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778761052.436146      22 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778761052.510951      22 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778761053.066077      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778761053.066121      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778761053.066124      22 computation_placer.cc:177] computation placer alr

Loading Perch model from: /kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/bird-vocalization-classifier/8


I0000 00:00:1778761080.896254      22 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1778761080.902342      22 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Loading metadata...
After quality filter: 22411 rows.
Extracting features (Check logs every 100 rows)...


  0%|          | 0/22411 [00:00<?, ?it/s]

I0000 00:00:1778761087.120245      71 service.cc:152] XLA service 0x7c948c008e60 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778761087.120326      71 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1778761087.120334      71 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1778761088.721200      71 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-05-14 12:18:15.090515: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-14 12:18:15.228210: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-14 12:18:15.422507: E external/local_xl

Extraction complete.
Perch shape: (60808, 1284), CV shape: (60808, 31)
Saved Perch features to /kaggle/working/features_v3_perch_discrete.parquet
Saved CV features to /kaggle/working/features_v3_cv_discrete.parquet
